# Module 2 · Lesson 01: Zero-Shot Prompting

**Zero-shot prompting** means asking the model to perform a task *without any examples*.
The model relies entirely on its training data and your instructions.

## What you will learn
1. Effective zero-shot prompt patterns
2. Output **format specification**
3. **Role/persona** assignment
4. Zero-shot **classification**
5. **Structured output** (JSON) extraction
6. Using **constraints** to control output

In [1]:
# ── Setup ──────────────────────────────────────────────
import os
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, Markdown
 
load_dotenv(Path.cwd().parent / ".env")
 
from openai import OpenAI
 
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
 
def ask(prompt, system=None, temperature=0.7, max_tokens=1200):
    """Helper: call GPT and return text."""
    msgs = []
    if system:
        msgs.append({"role": "system", "content": system})
    msgs.append({"role": "user", "content": prompt})
    r = client.chat.completions.create(
        model="gpt-4o-mini", 
        messages=msgs,
        temperature=temperature, 
        max_tokens=max_tokens
    )
    return r.choices[0].message.content
 
if client:
    print("Client is ready")
else:
    print("Check API key and OpenAI client")
 

Client is ready


---
## 1. Simple Zero-Shot

The simplest form — just ask directly:

In [2]:
result = ask("What is the capital of France?")
display(Markdown(f"**Q**: What is the capital of France\n\n**A**: {result}"))

**Q**: What is the capital of France

**A**: The capital of France is Paris.

---
## 2. Format Specification

Tell the model **exactly** what format you want:

In [3]:
prompt = """Extract the email address from this text and return ONLY the email, nothing else:

Hi, you can reach me at john.doe@example.com for more information.
"""

result = ask(prompt, max_tokens=50)
display(Markdown(f"**Extracted:** `{result.strip()}`"))

**Extracted:** `john.doe@example.com`

---
## 3. Role / Persona Assignment

The system prompt sets *who* the model should be:

In [4]:
result = ask(
    prompt="Explain what an API is to a non-technical person.",
    system="You are a professional technical writer who explains complex concepts simply."
)

display(Markdown(f"### Technical Writer\n\n {result}"))

### Technical Writer

 An API, or Application Programming Interface, is like a waiter in a restaurant. Imagine you go to a restaurant and want to order food. You don’t go to the kitchen and start cooking yourself; instead, you tell the waiter what you want, and they communicate your order to the kitchen. Once your food is prepared, the waiter brings it back to your table.

In the digital world, an API works the same way. It allows different software programs to communicate with each other. For example, when you use an app on your phone to check the weather, that app sends a request to a weather service using an API. The service then sends back the weather information, which the app shows to you.

So, in simple terms, an API is a set of rules and tools that lets different software talk to each other, just like how a waiter helps you get your food from the kitchen.

---
## 4. Zero-Shot Classification

LLMs are excellent **zero-shot classifiers**. No training data needed!

In [5]:
texts = [
    "I absolutely love this product! Best purchase ever",
    "The shipping was delayed and the item arrived damaged",
    "It's okay, nothing special but does the job"
]

print(f"{'Text':<55} {'Sentiment'}")
print("_" * 101)

for text in texts:
    prompt = f"Classify the sentiment as exactly one word: positive, negative or neutral.\n\nText:{text}\n\nClassification:"
    sentiment = ask(prompt, temperature=0, max_tokens=10).strip().lower()
    emoji = {"positive": "🟢", "negative": "🔴", "neutral": "🟡"}.get(sentiment, "⚪")
    print(f"{text[:52]+'...':<55} {emoji} {sentiment}")

Text                                                    Sentiment
_____________________________________________________________________________________________________
I absolutely love this product! Best purchase ever...   🟢 positive
The shipping was delayed and the item arrived damage... 🔴 negative
It's okay, nothing special but does the job...          🟡 neutral


# Role - Context - Structure


In [6]:
result = ask(
    prompt="Explain what an API is to a 10 years old child. Do it in 3 short sentences. Put them in ordered bullets.",
    system="You are my school teacher who loves analogies and explains everything in an easy way"
)

display(Markdown(f"### Technical Writer:\n\n{result}"))

### Technical Writer:

Sure! Here’s a simple way to understand what an API is:

1. Imagine you are at a restaurant, and you want to order food; the menu is like a list of options you can choose from.  
2. The waiter takes your order and brings it to the kitchen, which is like a computer that prepares your request.  
3. When the food is ready, the waiter brings it back to you, just like an API helps different software talk to each other and share information.  

> 💡 Use `temperature=0` for classification to get deterministic, consistent results.

---
## 5. Structured Output (JSON)

In [7]:
import json

prompt = """Extract information from this text and return as valid JSON:

Text: Meeting with Sarah Johnson scheduled for March 15, 2026 at 2:30PM to discuss the Q1 budget report.

Return JSON fields: attendee, date, time, topic

DO NOT wrap the output in markdown code fences or any other formating.
Return ONLY the raw JSON object, nothing else.

JSON:
"""

result = ask(prompt, temperature=0)
try:
    parsed = json.loads(result)
    display(Markdown(f"```json\n{json.dumps(parsed, indent=2)}\n```"))
except json.JSONDecodeError:
    print(f"Raw output: {result}")
    print("Invalid JSON")

```json
{
  "attendee": "Sarah Johnson",
  "date": "2026-03-15",
  "time": "14:30",
  "topic": "Q1 budget report"
}
```

In [8]:
response_json = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{"role":"user", "content": prompt}],
    response_format={"type":"json_object"}, # force valid JSON output
    temperature=0
)

new_res = response_json.choices[0].message.content

try:
    parsed = json.loads(new_res)
    display(Markdown(f"```json\n{json.dumps(parsed, indent=2)}\n```"))
except json.JSONDecodeError:
    print(f"Raw output: {new_res}")
    print("Invalid JSON")

```json
{
  "attendee": "Sarah Johnson",
  "date": "2026-03-15",
  "time": "14:30",
  "topic": "Q1 budget report"
}
```

In [9]:
import json

response_json = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{"role":"user", "content": "Return a short greeting and a lucky number"}],
    response_format={"type":"json_schema",
                     "json_schema":{
                         "name": "greeting_response",
                         "schema":{
                             "type":"object",
                             "properties": {
                                 "greeting": {"type":"string"},
                                 "luckyNumber": {"type":"integer"}
                             },
                             "required":["greeting", "luckyNumber"],
                             "additionalProperties": False
                         },
                         "strict":True
                     }
    }, 
    temperature=0
)

new_res = response_json.choices[0].message.content
parsed = json.loads(new_res)

print(json.dumps(parsed, indent=2))

{
  "greeting": "Hello! Wishing you a wonderful day!",
  "luckyNumber": 7
}


In [10]:
import json
 
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "Describe an API endpoint for user login."}
    ],
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "api_endpoint",
            "schema": {
                "type": "object",
                "properties": {
                    "endpoint": {"type": "string"},
                    "method": {
                        "type": "string",
                        "enum": ["GET", "POST", "PUT", "DELETE"]
                    },
                    "request_body": {
                        "type": "object",
                        "properties": {
                            "email": {"type": "string"},
                            "password": {"type": "string"}
                        },
                        "required": ["email", "password"],
                        "additionalProperties": False
                    },
                    "response": {
                        "type": "object",
                        "properties": {
                            "token": {"type": "string"},
                            "expires_in": {"type": "integer"}
                        },
                        "required": ["token", "expires_in"],
                        "additionalProperties": False
                    }
                },
                "required": ["endpoint", "method", "request_body", "response"],
                "additionalProperties": False
            },
            "strict": True
        }
    },
    temperature=0
)
 
parsed = json.loads(response.choices[0].message.content)
print(json.dumps(parsed, indent=2))

{
  "endpoint": "/api/login",
  "method": "POST",
  "request_body": {
    "email": "user@example.com",
    "password": "userpassword"
  },
  "response": {
    "token": "abc123xyz",
    "expires_in": 3600
  }
}


---
## 6. Constraints

Adding explicit **constraints** controls length, format, and content:

In [11]:
prompt = """Write a product description for a wireless mouse.
Constraints:
- Maximum 50 words
- Include at least one benefit
- Do not mention price
- End with a call to action

Description:
"""

result = ask(prompt)
word_count = len(result.split())

display(Markdown(f"**Result**: {result}\n\n **Word count**: {word_count}"))

**Result**: Experience seamless navigation with our sleek wireless mouse, designed for ultimate comfort and precision. Enjoy the freedom of movement without tangled cords, perfect for both work and play. Elevate your productivity today—grab yours and transform your computing experience!

 **Word count**: 38

# Multi-dimensional contraints (style - structure - semantics)

In [12]:
prompt = """
Write a product description for wireless mouse.

Contraints:
- More than 40 words less than 60
- Exactly 2 sentences
- First sentece: describe features
- Second sentece: emphasize a user benefit
- Include exactly 1 emoji
- Must contain the word "precision"
- Do NOT use passive voice
- End with a call to action

Return ONLY the description
"""

result = ask(prompt)

word_count = len(result.split())

display(Markdown(f"**Result**: {result}\n\n **Word count**: {word_count}"))


**Result**: Experience seamless navigation with our wireless mouse, featuring adjustable DPI settings for precision control, ergonomic design for comfort, and long battery life for uninterrupted use. Elevate your productivity and enjoy effortless scrolling – grab yours today! 🖱️

 **Word count**: 37

---
## 7. Prompt Gallery: Real-World System Prompts

Let's study system prompts from **real community projects**. Each uses a different technique
to get reliable, high-quality outputs.

| Source | Key Technique |
|--------|---------------|
| Gmail Summarizer | Structured rules + labels |
| Budget Travel Agent | Role + constraint + format |
| Biomedical Summariser | Audience awareness |
| Code Explainer | Section structure |

In [13]:
# ── Prompt Gallery: 4 real-world system prompts ─────────
 
gallery = {
    "Gmail Summarizer": {
        "prompt": """You summarize email threads. For each email:
- Subject line (max 10 words)
- Label: ACTION_REQUIRED | FYI | PROMO | URGENT
- Summary (max 2 sentences)
- Has link: yes/no
Return as a numbered list.""",
        "technique": "Structured rules with labels and constraints",
    },
    "Budget Travel Agent": {
        "prompt": """You are a budget travel advisor. For any destination:
1. List top 5 FREE attractions
2. Suggest 3 budget restaurants (under $15/meal)
3. Give one money-saving local tip
Respond in markdown with headers.""",
        "technique": "Role + numbered constraints + format (markdown)",
    },
    "Biomedical Summariser": {
        "prompt": """Summarize biomedical research articles for a mixed audience:
students, early researchers, and professionals.
- Use bullet points for key findings
- Highlight methodology and sample size
- Note limitations and future directions
Tone: professional, clear, accessible.""",
        "technique": "Audience awareness + structure + tone",
    },
    "Code Explainer": {
        "prompt": """You explain code to developers. Structure your response as:
1) Direct Answer (1-2 sentences)
2) Explanation (why it works)
3) Example (working code snippet)
4) Common Pitfalls (what to avoid)
5) Next Steps (what to learn next)""",
        "technique": "Section-structured output format",
    }
}
 
# Display each prompt with analysis
for name, info in gallery.items():
    print(f"\n{'=' * 60}")
    print(f"  {name}")
    print(f"  Technique: {info['technique']}")
    print(f"{'=' * 60}")
    print(info['prompt'])


  Gmail Summarizer
  Technique: Structured rules with labels and constraints
You summarize email threads. For each email:
- Subject line (max 10 words)
- Label: ACTION_REQUIRED | FYI | PROMO | URGENT
- Summary (max 2 sentences)
- Has link: yes/no
Return as a numbered list.

  Budget Travel Agent
  Technique: Role + numbered constraints + format (markdown)
You are a budget travel advisor. For any destination:
1. List top 5 FREE attractions
2. Suggest 3 budget restaurants (under $15/meal)
3. Give one money-saving local tip
Respond in markdown with headers.

  Biomedical Summariser
  Technique: Audience awareness + structure + tone
Summarize biomedical research articles for a mixed audience:
students, early researchers, and professionals.
- Use bullet points for key findings
- Highlight methodology and sample size
- Note limitations and future directions
Tone: professional, clear, accessible.

  Code Explainer
  Technique: Section-structured output format
You explain code to developers. 

In [14]:
travel_result = ask(
    prompt="I'm visiting Lisbon, Portugal for 3 days on a tight budget.",
    system=gallery["Budget Travel Agent"]["prompt"]
)

display(Markdown(f"### Budget Travel Agent Response\n\n{travel_result}"))

### Budget Travel Agent Response

# Budget Travel Guide to Lisbon, Portugal

## Top 5 FREE Attractions
1. **Alfama District**: Wander through the narrow, winding streets of one of Lisbon's oldest neighborhoods. Enjoy the stunning views from various miradouros (viewpoints) like Miradouro de Santa Luzia.
   
2. **Lisbon Cathedral (Sé de Lisboa)**: Visit this iconic cathedral for free. It's a great place to learn about Lisbon's history and architecture.

3. **Belém Tower**: While there is an entrance fee to go inside, you can enjoy the beautiful views of this UNESCO World Heritage site from the outside and explore the surrounding area.

4. **Praça do Comércio**: Stroll through this grand square that opens up to the Tagus River. It's the perfect spot for people-watching and absorbing the local atmosphere.

5. **Parque das Nações**: Explore the modern side of Lisbon with its beautiful waterfront, gardens, and public art installations. Perfect for a leisurely walk.

## Budget Restaurants (Under $15/meal)
1. **Time Out Market**: While some stalls can be pricey, there are many affordable options. Look for local favorites like the sandwiches from Manteigaria Silva.

2. **O Prego da Peixaria**: Known for its delicious seafood sandwiches, this casual eatery offers a variety of prego (steak) and fish options at reasonable prices.

3. **Taberna da Rua das Flores**: A cozy tavern that serves traditional Portuguese dishes. Dishes are small and affordable, making it easy to try a variety of flavors.

## Money-Saving Local Tip
**Use the Lisbon Card**: If you plan on visiting several attractions (even if some are paid), consider getting a Lisbon Card. It offers free entry to many attractions, discounts on others, and free public transportation, which can save you money over your 3-day stay.

In [15]:
gmail_result = ask(
    """Here's an email thread:
From: marketing@company.com
Subject: Re: Q2 Campaign Launch — Asset Review Needed
Body: Hi team, please review the attached creatives for the Q2 social campaign.
We need approvals by Friday. The campaign landing page is at https://company.com/q2launch.
Let me know if any changes are needed. Thanks!""",
    system=gallery["Gmail Summarizer"]["prompt"]
)
display(Markdown(f"### Gmail Summarizer Response\n\n{gmail_result}"))
print("\nNotice the consistent labeling and structured bullet format!")


### Gmail Summarizer Response

1. Subject line: Q2 Campaign Launch — Asset Review Needed  
   Label: ACTION_REQUIRED  
   Summary: The team is requested to review the attached creatives for the Q2 social campaign and provide approvals by Friday. The campaign landing page is linked for reference.  
   Has link: yes


Notice the consistent labeling and structured bullet format!


In [16]:
bio_result = ask(
    """A 2024 study published in The Lancet examined the efficacy of a novel mRNA-based
therapeutic vaccine for stage III melanoma. The randomized, double-blind trial enrolled
340 patients across 22 clinical sites. Results showed a 44% reduction in recurrence risk
(HR 0.56, 95% CI 0.40–0.78, p=0.0007) over a 24-month follow-up. Common adverse effects
included fatigue (32%) and injection-site reactions (28%). The authors noted the relatively
short follow-up period and homogeneous population (predominantly Caucasian, median age 58)
as key limitations, and called for larger Phase III trials with diverse cohorts.""",
    system=gallery["Biomedical Summariser"]["prompt"]
)
display(Markdown(f"### Biomedical Summariser Response\n\n{bio_result}"))
print("\nNotice how it highlights methodology, limitations, and stays accessible!")

### Biomedical Summariser Response

### Key Findings from the Study on mRNA-Based Therapeutic Vaccine for Stage III Melanoma

- **Study Overview**: 
  - Type: Randomized, double-blind trial
  - Sample Size: 340 patients
  - Locations: Conducted across 22 clinical sites

- **Efficacy Results**:
  - The novel mRNA-based therapeutic vaccine demonstrated a **44% reduction in recurrence risk**.
  - Hazard Ratio (HR): 0.56 with a 95% Confidence Interval (CI) of 0.40–0.78.
  - Statistical significance: p=0.0007 over a **24-month follow-up period**.

- **Adverse Effects**:
  - Common side effects reported included:
    - Fatigue: 32%
    - Injection-site reactions: 28%

### Methodology
- The study employed a rigorous randomized, double-blind design to ensure unbiased results.
- Patients were monitored for recurrence over a 24-month period post-vaccination.

### Limitations
- **Follow-Up Duration**: The 24-month follow-up period may not provide sufficient long-term efficacy data.
- **Population Homogeneity**: The study cohort was predominantly Caucasian with a median age of 58, which may limit the generalizability of the results to more diverse populations.

### Future Directions
- The authors advocate for **larger Phase III trials** that include more diverse patient populations to validate the findings and assess long-term outcomes.
- Further research is needed to explore the vaccine's efficacy across different demographics and to monitor long-term safety and effectiveness beyond the initial follow-up period.


Notice how it highlights methodology, limitations, and stays accessible!


In [17]:
code_result = ask(
    """Explain this Python code:
result = {k: v for k, v in sorted(data.items(), key=lambda item: item[1], reverse=True)[:5]}""",
    system=gallery["Code Explainer"]["prompt"]
)
display(Markdown(f"### Code Explainer Response\n\n{code_result}"))
print("\nNotice the 5-section structure: Answer → Explanation → Example → Pitfalls → Next Steps!")

### Code Explainer Response

1) **Direct Answer:** This Python code creates a dictionary called `result` that contains the top 5 key-value pairs from the `data` dictionary, sorted by their values in descending order.

2) **Explanation:** The code uses a dictionary comprehension to iterate over the items of the `data` dictionary. The `sorted()` function sorts these items based on their values (the second element of each item) in descending order due to `reverse=True`. The slicing `[:5]` ensures that only the top 5 items are taken, and the comprehension constructs a new dictionary from these items.

3) **Example:**
   ```python
   data = {'a': 10, 'b': 20, 'c': 5, 'd': 15, 'e': 25, 'f': 30}
   result = {k: v for k, v in sorted(data.items(), key=lambda item: item[1], reverse=True)[:5]}
   print(result)  # Output: {'f': 30, 'e': 25, 'b': 20, 'd': 15, 'a': 10}
   ```

4) **Common Pitfalls:** One common pitfall is assuming that `data` contains at least 5 items. If it has fewer than 5 elements, the code will simply return all available items without error, which may not be the intended behavior. Another pitfall is not understanding that the sorting is based on values, which may lead to confusion if keys are expected to be sorted instead.

5) **Next Steps:** To deepen your understanding, explore how to handle ties in value sorting (e.g., using a secondary key), learn about other data structures like `Counter` from the `collections` module for similar tasks, or investigate performance considerations when working with large dictionaries.


Notice the 5-section structure: Answer → Explanation → Example → Pitfalls → Next Steps!


> **Exercise:** Pick a business domain you're interested in (e.g., fitness coaching,
> legal review, recipe generation) and write your own system prompt following the
> patterns above. Test it with 3 different user queries.

# Exercise

In [29]:
prompt = """ Εξήγαγε πληροφορίες από την επόμενη πρόταση και επίστρεψε την πληροφορία σε έγκυρη JSON μορφή:

Ο Γιάννης αγόρασε ένα κινητό και είναι πολύ ευχαριστημένος με την κάμερα αλλά όχι με την μπαταρία.

Πεδία JSON: sentiment, positive_aspects, negative_aspects

ΜΗ βάλεις το αποτέλεσμα σε σημεία στίξης και επέστρεψε μόνο τη raw JSON μορφή. Γράψε στα αγγλικά.
"""

result = ask(prompt, temperature=0)
try:
    parsed = json.loads(result)
   # print(f"Raw output: {result}")
    display(Markdown(f"```json\n{json.dumps(parsed, indent=2)}\n```"))
except json.JSONDecodeError:
    print(f"Raw output: {result}")
    print("Invalid JSON")

```json
{
  "sentiment": "mixed",
  "positive_aspects": [
    "camera"
  ],
  "negative_aspects": [
    "battery"
  ]
}
```

---
## Key Takeaways 📝

| Technique | When to Use |
|-----------|------------|
| **Direct question** | Simple factual queries |
| **Format specification** | When you need specific output format |
| **Role assignment** | To control tone, expertise, style |
| **Classification** | Categorizing text (use temp=0) |
| **JSON extraction** | Structured data from unstructured text |
| **Constraints** | Controlling length, style, content boundaries |
| **Prompt gallery** | Study real prompts to develop critical analysis skills |

### Zero-Shot Tips
1. Be **specific** about what you want
2. Specify the **output format** explicitly
3. Use **roles/personas** to guide behaviour
4. Add **negative constraints** (what NOT to do)
5. Use `temperature=0` for deterministic tasks

---
**Next:** `02_few_shot_examples.ipynb` — Improve results by providing examples